# 삼성디스플레이 AI Agent Architecture 과정
# 2일차 강사용 노트북
## 제조 기술 문서 참조형 RAG Agent v1 설계·실행·Trace 분석

---

2일차는 RAG 코딩 수업이 아니라,
**제조 기술 문서를 Agent가 검색 가능한 지식 구조로 만들고,
검색 결과를 State·Node·Trace로 연결하는 아키텍처 실습**입니다.

## 이 노트북의 목적

이 노트북은 `src/day2`의 전체 코드를 복사한 파일이 **아닙니다**.
2일차 RAG Agent v1 실습의 **핵심 흐름을 하나로 연결해 보기 위한 강사용 실행형 요약 노트북**입니다.

실제 프로젝트 기준 파일은 `src/day2` 폴더의 Python 파일입니다.
이 노트북은 강의 중 전체 구조를 설명하고, 핵심 코드를 직접 실행해보며,
산출물과 Trace를 함께 확인하기 위한 보조 자료입니다.

노트북 안의 코드는 원본 `src/day2` 파일의 모든 기능을 그대로 복사한 것이 아니라,
**강의 흐름을 이해하기 위해 핵심 처리 흐름만 간결하게 재구성한 코드**입니다.
따라서 원본과 세부 구현(예: 실제 LangGraph StateGraph, Mustache 템플릿, llm_client 호출)이 다를 수 있습니다.

> 실행 메모: 이 노트북은 위에서 아래로 한 번에 실행되며(Run All), 외부 패키지(LangGraph 등) 없이도 동작합니다.
> 셀을 순서대로 실행하면 `outputs/day2`의 산출물이 노트북 로직 기준으로 갱신됩니다(파일명·경로는 그대로 유지).

## src/day2 원본 파일과 노트북 반영 범위

| 원본 파일 | 노트북 반영 방식 | 강의에서 강조할 핵심 |
|---|---|---|
| `src/day2/rag_document_loader.py` | 핵심 문서 로드 로직을 노트북 셀로 재구성 | RAG 대상 문서 입력 구조 검증 |
| `src/day2/chunk_builder.py` | 핵심 chunk/metadata 생성 로직을 노트북 셀로 재구성 | 검색 가능한 지식 단위 설계, metadata 생성 |
| `src/day2/chroma_index_builder.py` | Chroma/Ollama 인덱스 생성 흐름을 노트북 셀로 재구성 | Chroma Vector DB 인덱스 생성, 의미 기반 검색 준비 |
| `src/day2/rag_search.py` | Top-3 검색 로직을 노트북 셀로 재구성 | Top-3 근거 후보, 검색 품질 검토, `search_manual` Tool 기반 |
| `src/day2/graph_state.py` | State 구조와 변화 예시를 노트북 셀로 재구성 | Node 사이에서 공유되는 업무 처리 문맥 |
| `src/day2/langgraph_rag_graph_runner.py` | Node/조건부 분기 흐름을 mock graph로 재구성 | Node 실행 순서, 조건부 분기, grounding 검증 |
| `src/day2/day2_rag_agent_v1.py` | 최종 RAG Agent v1 흐름을 노트북 셀로 요약 실행 | RAG Agent v1 통합 실행 결과, 3일차 MCP 연결 |

## 오늘의 핵심 흐름

```
입력 데이터 확인
  → 문서 로드
    → chunk / metadata
      → Chroma / Ollama 인덱스 생성
        → Top-3 검색
          → State 저장
            → Node / 조건부 분기
              → Trace 확인
                → RAG Agent v1 최종 결과
                  → 3일차 MCP Tool 연결
```

| 단계 | 참조 원본 파일 | 노트북 실행 핵심 코드 | 확인 입력 데이터·산출물 | 다음 단계와의 연결 |
|---|---|---|---|---|
| 입력 데이터 확인 | (입력 데이터) | 문서·질의 미리보기 | `docs/*.md`, `data/sample_*` | RAG 흐름에 들어가는 입력 확인 |
| 문서 로드 | `rag_document_loader.py` | `load_documents()` | `document_load_result.md` | 문단이 chunk 분할 입력이 됨 |
| chunk / metadata | `chunk_builder.py` | `build_chunks()` | `chunk_preview.json`, `chunk_build_result.md` | chunk가 검색 대상 지식 단위가 됨 |
| Chroma 인덱스 | `chroma_index_builder.py` | `build_chroma_index()` | `chroma_build_result.md` | 의미 기반 검색 준비(불가 시 fallback) |
| Top-3 검색 | `rag_search.py` | `search_top_k_nb()` | `rag_test_results.md` | 결과가 State.retrieved_docs로 전달됨 |
| State 저장 | `graph_state.py` | `create_initial_state()` | `state_demo_result.md` | retrieved_docs가 답변·grounding 입력 |
| Node / 조건부 분기 | `langgraph_rag_graph_runner.py` | `run_mock_graph()` | `langgraph_rag_trace.md` | 근거 부족 시 query_rewrite로 분기 |
| RAG Agent v1 결과 | `day2_rag_agent_v1.py` | 최종 리포트 조립 | `day2_rag_agent_v1_result.md` | 3일차 search_manual MCP Tool로 분리 |

## 0. 준비: 공통 유틸리티

노트북 위치에 상관없이 **프로젝트 루트를 자동 탐색**하고, 입력/산출물 확인·저장을 돕는 함수를 정의합니다.
프로젝트 경로는 하드코딩하지 않으며, 파일이 없어도 traceback 대신 안내 메시지로 대체해 **Run All이 중단되지 않게** 합니다.

In [ ]:
# 공통 유틸리티: 프로젝트 루트 탐색 + 입력/산출물 읽기·저장 헬퍼
from pathlib import Path
import json
import re
import urllib.request
import urllib.error
from collections import Counter
from IPython.display import Markdown, display


def find_project_root(markers=("src", "docs", "outputs")):
    """현재 폴더에서 시작해 상위로 올라가며 src/docs/outputs가 모두 있는 위치를 루트로 판단."""
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        if all((base / name).is_dir() for name in markers):
            return base
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    print("프로젝트 루트를 찾지 못했습니다. 현재 폴더:", Path.cwd())
    print("src, docs, outputs 폴더가 함께 있는 위치(프로젝트 루트)에서 실행해 주세요.")
else:
    print("프로젝트 루트:", PROJECT_ROOT)


def _resolve(rel):
    return None if PROJECT_ROOT is None else (PROJECT_ROOT / rel)


def _missing(rel):
    display(Markdown(f"> `{rel}` 파일을 찾지 못했습니다. 앞 단계 셀을 먼저 실행했는지 확인하세요."))


def esc(value):
    """Markdown 표 셀이 깨지지 않도록 줄바꿈/파이프를 치환합니다."""
    return str(value if value is not None else "").replace("\n", " ").replace("|", "/")


def read_text(rel):
    path = _resolve(rel)
    if path is None or not path.exists():
        _missing(rel)
        return None
    return path.read_text(encoding="utf-8-sig")


def show_markdown(rel):
    text = read_text(rel)
    if text is not None:
        display(Markdown(text))


def preview_text(rel, n=25):
    """텍스트/Markdown 파일 앞부분 n줄만 미리보기(전체 출력 금지)."""
    text = read_text(rel)
    if text is None:
        return
    lines = text.splitlines()
    print(f"[{rel}] 앞 {min(n, len(lines))}줄 미리보기 (총 {len(lines)}줄)\n")
    print("\n".join(lines[:n]))


def search_paragraphs(rel, keyword, max_hits=3):
    """키워드가 포함된 문단을 검색해 일부만 보여줍니다."""
    text = read_text(rel)
    if text is None:
        return
    paras = [p.strip() for p in text.split("\n\n") if keyword in p]
    if not paras:
        print(f"[{rel}] '{keyword}' 포함 문단을 찾지 못했습니다.")
        return
    print(f"[{rel}] '{keyword}' 포함 문단 {min(max_hits, len(paras))}/{len(paras)}개\n")
    for para in paras[:max_hits]:
        print("-" * 40)
        print(para)


def preview_json(rel, limit=3, fields=None, list_key=None):
    """JSON 산출물의 앞부분 일부 항목만 출력(전체 출력 금지)."""
    text = read_text(rel)
    if text is None:
        return
    data = json.loads(text)
    items = data
    if list_key and isinstance(data, dict):
        print("최상위 키:", list(data.keys()))
        items = data.get(list_key, [])
    if isinstance(items, list):
        print(f"총 {len(items)}개 항목 중 앞 {min(limit, len(items))}개만 표시합니다.")
        preview = items[:limit]
        if fields:
            preview = [{key: item.get(key, "") for key in fields} for item in preview]
        for item in preview:
            display(item)
    else:
        display(items)


def show_json_fields(rel, fields=None):
    text = read_text(rel)
    if text is None:
        return
    data = json.loads(text)
    if isinstance(data, dict) and fields:
        display({key: data.get(key, "") for key in fields if key in data} or data)
    else:
        display(data)


def save_markdown(rel, text):
    path = _resolve(rel)
    if path is None:
        print("프로젝트 루트를 찾지 못해 저장을 건너뜁니다:", rel)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8-sig")
    print("저장 완료:", rel)


def save_json(rel, obj):
    path = _resolve(rel)
    if path is None:
        print("프로젝트 루트를 찾지 못해 저장을 건너뜁니다:", rel)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8-sig")
    print("저장 완료:", rel)

## 1. 입력 데이터 미리보기

**참조 입력 데이터:** `docs/alarm_manual.md`, `docs/troubleshooting_guide.md`, `docs/quality_standard.md`, `data/sample_rag_queries.json`, `data/sample_query.json`

2일차 RAG Agent v1은 교육용 제조 기술 문서와 샘플 질의를 입력으로 사용합니다.
입력 데이터를 먼저 확인하면 이후 chunk, metadata, 검색 결과, State 흐름을 이해하기 쉽습니다.
단, 문서 전체를 출력하면 복잡하므로 **일부 문단과 일부 질의만** 확인합니다.

In [ ]:
# 교육용 제조 기술 문서 앞부분만 미리보기 (전체 출력하지 않음)
for doc in ("docs/alarm_manual.md", "docs/troubleshooting_guide.md", "docs/quality_standard.md"):
    print("=" * 60)
    preview_text(doc, n=25)
    print()

# ALM-TEMP-402 키워드가 포함된 문단만 검색해 확인
print("=" * 60)
search_paragraphs("docs/alarm_manual.md", "ALM-TEMP-402", max_hits=3)

In [ ]:
# 샘플 질의: sample_rag_queries.json은 queries 배열 앞 3개만 확인
preview_json(
    "data/sample_rag_queries.json",
    limit=3,
    list_key="queries",
    fields=["id", "user_query", "intent", "difficulty"],
)
print()
# sample_query.json은 단일 시나리오 객체이므로 주요 필드만 확인
show_json_fields("data/sample_query.json", fields=["scenario_name", "user_query", "data_notice"])

## 2. 문서 로드 핵심 코드 실행

- **참조 원본 파일:** `src/day2/rag_document_loader.py`
- **생성 산출물:** `outputs/day2/document_load_result.md`

문서 로드는 단순 파일 읽기가 아니라, **RAG 대상 문서 입력 구조를 검증하는 단계**입니다.
문서 수·문단 수·분량을 확인해 이후 chunk 분할의 입력이 정상인지 점검합니다.

> 원본 `rag_document_loader.py`의 핵심 흐름만 재구성했습니다(세부 구현은 원본과 다를 수 있음).

In [ ]:
# 핵심 문서 로드 로직 (원본 흐름 재구성)
DOC_FILES = ["docs/alarm_manual.md", "docs/troubleshooting_guide.md", "docs/quality_standard.md"]


def load_documents(doc_files):
    """docs 폴더의 Markdown 문서를 읽어 파일명/글자 수/문단 수로 입력 구조를 검증합니다."""
    docs = []
    for rel in doc_files:
        text = read_text(rel)
        if text is None:
            continue
        paragraphs = [b.strip() for b in text.split("\n\n") if b.strip()]
        docs.append({
            "file_name": Path(rel).name,
            "char_count": len(text),
            "paragraph_count": len(paragraphs),
            "paragraphs": paragraphs,
        })
    return docs


DOCS = load_documents(DOC_FILES)

lines = ["# Day2 Markdown 문서 로드 결과", "", "## 1. 로드 요약", "",
         f"- 읽은 문서 수: {len(DOCS)}",
         f"- 전체 문단 수: {sum(d['paragraph_count'] for d in DOCS)}",
         f"- 전체 글자 수: {sum(d['char_count'] for d in DOCS)}", "", "## 2. 문서별 요약", ""]
for d in DOCS:
    lines += [f"### {d['file_name']}", "",
              f"- 파일명: {d['file_name']}", f"- 글자 수: {d['char_count']}", f"- 문단 수: {d['paragraph_count']}", ""]
lines += ["## 3. 다음 단계 안내", "", "다음 단계에서는 문단을 chunk로 나누고 metadata를 생성합니다.", ""]

save_markdown("outputs/day2/document_load_result.md", "\n".join(lines))
show_markdown("outputs/day2/document_load_result.md")

## 3. Chunk와 Metadata 핵심 코드 실행

- **참조 원본 파일:** `src/day2/chunk_builder.py`
- **생성 산출물:** `outputs/day2/chunk_preview.json`, `outputs/day2/chunk_build_result.md`

chunk는 **검색 가능한 지식 단위**입니다.
`chunk_id`와 `doc_name`은 검색 결과와 Trace에서 **근거 위치를 추적하는 기준**입니다.
metadata는 **검색·필터링·3일차 Tool 입력값으로 재사용**됩니다.

> 원본의 키워드 사전·정규식 기반 metadata 추출 흐름을 재구성했습니다(세부 구현은 원본과 다를 수 있음).

In [ ]:
# 핵심 chunk/metadata 생성 로직 (원본 흐름 재구성)
MIN_PARAGRAPH_LENGTH = 15  # 너무 짧은 문단은 검색 근거로 부적합하여 제외

EQUIPMENT_TYPE_KEYWORDS = ["증착 설비", "검사 설비", "세정 설비", "이송 설비", "패널 제조 설비", "설비"]
PROCESS_NAME_KEYWORDS = ["증착 공정", "검사 공정", "세정 공정", "이송 공정", "공정 상태", "제조 라인", "디스플레이 패널 제조 라인"]
SYMPTOM_KEYWORDS = ["온도 상승", "반복 알람", "압력 불안정", "진공 상태", "냉각 상태", "공정 부하", "센서 값 변동", "검사 결과 변화"]
QUALITY_METRIC_KEYWORDS = ["품질 지표", "불량률", "수율", "검사 결과", "품질 영향", "품질 이상", "검사 공정"]
ACTION_KEYWORDS = ["확인 필요", "점검 필요", "추가 검토", "조치 방향", "원인 후보", "정비 이력", "설정 변경", "근거 확인", "품질 영향 여부 확인"]


def _uniq_join(items):
    out = []
    for it in items:
        if it and it not in out:
            out.append(it)
    return ", ".join(out)


def extract_metadata(text):
    """알람 코드/설비 ID는 정규식, 나머지는 교육용 키워드 사전으로 추출합니다."""
    alarm = _uniq_join(re.findall(r"\bALM-[A-Z]+-[0-9]+\b", text))
    equip = _uniq_join(re.findall(r"\bEQP-[A-Z]+-[0-9]+\b", text))
    etype = _uniq_join([k for k in EQUIPMENT_TYPE_KEYWORDS if k in text])
    proc = _uniq_join([k for k in PROCESS_NAME_KEYWORDS if k in text])
    symp = _uniq_join([k for k in SYMPTOM_KEYWORDS if k in text])
    qual = _uniq_join([k for k in QUALITY_METRIC_KEYWORDS if k in text])
    act = _uniq_join([k for k in ACTION_KEYWORDS if k in text])
    keywords = []
    for value in [alarm, equip, etype, proc, symp, qual, act]:
        for part in [x.strip() for x in value.split(",")]:
            if part and part not in keywords:
                keywords.append(part)
    return {"alarm_code": alarm, "equipment_id": equip, "equipment_type": etype,
            "process_name": proc, "symptom": symp, "quality_metric": qual,
            "action": act, "keywords": ", ".join(keywords)}


def build_chunks(doc_files):
    """문단을 검색 가능한 chunk로 나누고 metadata를 붙입니다. 제목(#)은 section_title로만 사용."""
    records = []
    n = 1
    for rel in doc_files:
        text = read_text(rel)
        if text is None:
            continue
        doc_name = Path(rel).name
        section = ""
        for block in text.split("\n\n"):
            para = block.strip()
            if not para:
                continue
            if re.match(r"^#{1,3}\s+", para):
                section = re.sub(r"^#{1,3}\s+", "", para).strip()
                continue
            if len(para.replace(" ", "").replace("\n", "")) < MIN_PARAGRAPH_LENGTH:
                continue
            meta = extract_metadata(para)
            records.append({"chunk_id": f"CHUNK-{n:04d}", "doc_name": doc_name, "section_title": section, **meta, "text": para})
            n += 1
    return records


CHUNKS = build_chunks(DOC_FILES)
save_json("outputs/day2/chunk_preview.json", CHUNKS)


def _short(text, max_len=120):
    one_line = " ".join(str(text or "").split())
    return one_line if len(one_line) <= max_len else one_line[:max_len] + "..."


doc_counter = Counter(r["doc_name"] for r in CHUNKS)
mlines = ["# Day2 Chunk 생성 결과", "", "## 1. 생성 요약", "",
          f"- 생성된 chunk 수: {len(CHUNKS)}", "", "## 2. 문서별 chunk 수", "",
          "| 문서명 | chunk 수 |", "|---|---:|"]
for dn, cnt in sorted(doc_counter.items()):
    mlines.append(f"| {dn} | {cnt} |")
mlines += ["", "## 3. metadata 추출 예시 (앞 5개)", ""]
for r in CHUNKS[:5]:
    mlines += [f"### {r['chunk_id']}", "",
               f"- doc_name: {r['doc_name']}", f"- section_title: {r['section_title']}",
               f"- alarm_code: {r['alarm_code']}", f"- equipment_id: {r['equipment_id']}",
               f"- keywords: {r['keywords']}", f"- text 미리보기: {_short(r['text'])}", ""]
save_markdown("outputs/day2/chunk_build_result.md", "\n".join(mlines))

# chunk_preview.json은 앞 3개만 핵심 필드 중심으로 확인
preview_json("outputs/day2/chunk_preview.json", limit=3,
             fields=["chunk_id", "doc_name", "section_title", "alarm_code", "equipment_id", "keywords", "text"])
print()
show_markdown("outputs/day2/chunk_build_result.md")

## 4. Chroma/Ollama 인덱스 생성 핵심 코드 실행

- **참조 원본 파일:** `src/day2/chroma_index_builder.py`
- **생성 산출물:** `outputs/day2/chroma_build_result.md`

Chroma는 문서 chunk를 vector 형태로 저장하고 **의미 기반 검색**을 가능하게 하는 Vector DB입니다.
Ollama embedding 모델은 문장의 의미를 벡터로 바꾸는 데 사용됩니다.
2일차에서는 검색 결과가 State와 Trace에 연결되는 구조를 이해하는 것이 핵심입니다.

> **중요:** Chroma/Ollama 환경이 준비되지 않아도 이 셀은 중단되지 않습니다.
> `chromadb` 미설치, Ollama 미실행, `nomic-embed-text` 모델 없음 등의 경우
> 짧은 안내만 출력하고, 이후 Top-3 검색은 **키워드 기반 fallback 검색**으로 진행합니다.
> 오류를 길게 디버깅하지 않는 것이 수업 운영의 기준입니다.

In [ ]:
# Chroma/Ollama 인덱스 생성 시도 (실패 시 fallback 안내)
COLLECTION_NAME = "manufacturing_rag_docs"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text:latest"
OLLAMA_EMBEDDING_URL = "http://localhost:11434/api/embeddings"

CHROMA_READY = False
CHROMA_COLLECTION = None


def ollama_embed(text, timeout=60):
    """Ollama embedding API를 호출해 숫자 벡터를 받습니다."""
    body = json.dumps({"model": OLLAMA_EMBEDDING_MODEL, "prompt": str(text or "")}, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(OLLAMA_EMBEDDING_URL, data=body, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    emb = data.get("embedding")
    if not isinstance(emb, list) or not emb:
        raise RuntimeError("Ollama 응답에 embedding 필드가 없습니다.")
    return [float(v) for v in emb]


def build_chroma_index():
    """chunk를 embedding해 Chroma collection에 저장합니다. 환경 미준비 시 fallback 안내."""
    global CHROMA_READY, CHROMA_COLLECTION
    try:
        import chromadb
    except Exception:
        print("chromadb 미설치 → 키워드 기반 fallback 검색을 사용합니다. (설치: pip install chromadb)")
        return
    try:
        ollama_embed("연결 확인", timeout=5)
    except Exception:
        print("Ollama embedding 호출 불가 → 키워드 기반 fallback 검색을 사용합니다.")
        print("준비 명령: ollama serve / ollama pull nomic-embed-text")
        return
    try:
        class _EmbeddingFunction:
            def name(self):
                return "simple_ollama_nomic_embed_text"

            def __call__(self, input):
                return [ollama_embed(t) for t in input]

        chroma_dir = _resolve("vector_db/chroma_db")
        chroma_dir.mkdir(parents=True, exist_ok=True)
        client = chromadb.PersistentClient(path=str(chroma_dir))
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass
        meta_keys = ("chunk_id", "doc_name", "section_title", "alarm_code", "equipment_id",
                     "equipment_type", "process_name", "symptom", "quality_metric", "action", "keywords")
        collection = client.create_collection(name=COLLECTION_NAME, embedding_function=_EmbeddingFunction())
        if CHUNKS:
            collection.add(
                documents=[c["text"] for c in CHUNKS],
                ids=[c["chunk_id"] for c in CHUNKS],
                metadatas=[{k: c.get(k, "") for k in meta_keys} for c in CHUNKS],
            )
        CHROMA_COLLECTION = collection
        CHROMA_READY = True
        print(f"Chroma 인덱스 생성 완료: {collection.count()}개 chunk 저장")
    except Exception as exc:
        print(f"Chroma 인덱스 생성 실패({type(exc).__name__}) → 키워드 기반 fallback 검색을 사용합니다.")


build_chroma_index()

clines = ["# Day2 Chroma Vector DB 생성 결과", "", "## 1. 생성 요약", "",
          f"- 입력 chunk 수: {len(CHUNKS)}",
          f"- Chroma 인덱스 사용 가능 여부: {CHROMA_READY}",
          f"- Embedding provider: Ollama ({OLLAMA_EMBEDDING_MODEL})",
          f"- collection 이름: {COLLECTION_NAME}", "",
          "## 2. 안내", "",
          "- Chroma/Ollama가 준비되면 의미 기반 검색을 사용합니다.",
          "- 준비되지 않으면 키워드 기반 fallback 검색으로 진행합니다(수업 흐름 유지, Saved Result Review).", ""]
save_markdown("outputs/day2/chroma_build_result.md", "\n".join(clines))
show_markdown("outputs/day2/chroma_build_result.md")

## 5. Top-3 RAG 검색 핵심 코드 실행

- **참조 원본 파일:** `src/day2/rag_search.py`
- **생성 산출물:** `outputs/day2/rag_test_results.md`
- **3일차 연결:** `search_manual` MCP Tool의 기반

Top-3 결과는 정답 3개가 아니라 **답변 전 검토할 근거 후보 집합**입니다.
`score`만 보지 말고 `doc_name`, `chunk_id`, `section_title`, `text`를 함께 확인합니다.
이 검색 기능은 3일차에 **search_manual MCP Tool**로 분리됩니다.

> Chroma가 준비되면 의미 기반 검색을, 아니면 **키워드 기반 fallback 검색**을 사용합니다.
> 어떤 경우에도 노트북은 중단되지 않습니다. fallback score는 단순 키워드 일치 기반 참고값입니다.

In [ ]:
# Top-3 검색 로직 (Chroma 가능 시 의미 기반, 아니면 키워드 fallback)
def _distance_to_score(distance):
    try:
        d = float(distance)
    except (TypeError, ValueError):
        return 0.0
    if d < 0:
        d = 0.0
    return round(1 / (1 + d), 4)


def _preview(text, max_len=180):
    one_line = " ".join(str(text or "").split())
    return one_line if len(one_line) <= max_len else one_line[:max_len] + "..."


def _keyword_search(query, chunks, top_k=3):
    """Chroma/Ollama가 없을 때 사용하는 단순 키워드 일치 기반 fallback 검색."""
    terms = [w for w in re.split(r"\s+", query.strip()) if len(w) >= 2]
    codes = re.findall(r"\b(?:ALM|EQP)-[A-Z]+-[0-9]+\b", query)
    scored = []
    for c in chunks:
        hay = c.get("text", "") + " " + c.get("keywords", "") + " " + c.get("section_title", "")
        hits = sum(1 for t in terms if t in hay)
        bonus = sum(2 for code in codes if code in hay)
        raw = hits + bonus
        if raw <= 0:
            continue
        denom = max(len(terms) + 2 * len(codes), 1)
        scored.append((round(raw / denom, 4), c))
    scored.sort(key=lambda x: x[0], reverse=True)
    out = []
    for i, (score, c) in enumerate(scored[:top_k]):
        out.append({"rank": i + 1, "score": score, "distance": round(1 - score, 4),
                    "chunk_id": c["chunk_id"], "doc_name": c["doc_name"], "section_title": c["section_title"],
                    "alarm_code": c.get("alarm_code", ""), "equipment_id": c.get("equipment_id", ""),
                    "keywords": c.get("keywords", ""), "text": c["text"], "preview": _preview(c["text"])})
    return out


def _chroma_search(query, top_k=3):
    res = CHROMA_COLLECTION.query(query_embeddings=[ollama_embed(query)], n_results=top_k)
    ids = (res.get("ids") or [[]])[0]
    docs = (res.get("documents") or [[]])[0]
    metas = (res.get("metadatas") or [[]])[0]
    dists = (res.get("distances") or [[]])[0]
    out = []
    for i, cid in enumerate(ids):
        m = metas[i] if i < len(metas) and metas[i] else {}
        dist = dists[i] if i < len(dists) else None
        text = docs[i] if i < len(docs) else ""
        out.append({"rank": i + 1, "score": _distance_to_score(dist),
                    "distance": round(float(dist), 4) if dist is not None else 0.0,
                    "chunk_id": str(cid), "doc_name": str(m.get("doc_name", "")),
                    "section_title": str(m.get("section_title", "")), "alarm_code": str(m.get("alarm_code", "")),
                    "equipment_id": str(m.get("equipment_id", "")), "keywords": str(m.get("keywords", "")),
                    "text": str(text), "preview": _preview(text)})
    return out


def search_top_k_nb(query, top_k=3):
    """질문을 받아 Top-K 근거 후보를 돌려줍니다. 이 dict 구조가 State.retrieved_docs 한 건입니다."""
    if not query or not query.strip():
        return []
    if CHROMA_READY and CHROMA_COLLECTION is not None:
        try:
            return _chroma_search(query, top_k)
        except Exception as exc:
            print(f"Chroma 검색 실패({type(exc).__name__}) → 키워드 fallback으로 진행합니다.")
    return _keyword_search(query, CHUNKS, top_k)


def load_sample_queries():
    text = read_text("data/sample_rag_queries.json")
    queries = []
    if text is not None:
        data = json.loads(text)
        queries = data.get("queries", []) if isinstance(data, dict) else (data if isinstance(data, list) else [])
    if not queries:
        queries = [
            {"id": "Q-EDU-1", "user_query": "ALM-TEMP-402 반복 알람의 의미와 확인 항목", "intent": "알람 의미 확인"},
            {"id": "Q-EDU-2", "user_query": "EQP-EV-03 온도 상승 반복 알람의 원인 후보", "intent": "원인 후보"},
        ]
    return queries


SAMPLE_QUERIES = load_sample_queries()
SEARCH_RESULTS = []
for q in SAMPLE_QUERIES[:3]:
    uq = str(q.get("user_query", "") or "")
    SEARCH_RESULTS.append({"query_id": q.get("id", ""), "user_query": uq, "results": search_top_k_nb(uq, 3)})

rlines = ["# Day2 RAG 검색 결과 (Top-3 근거 후보)", "",
          f"- 검색 방식: {'Chroma Vector DB (의미 기반)' if CHROMA_READY else '키워드 기반 fallback'}", "",
          "> Top-3는 정답 3개가 아니라 답변 전 검토할 근거 후보 집합입니다. "
          "score, doc_name, chunk_id, section_title, text를 함께 확인하세요.", ""]
for item in SEARCH_RESULTS:
    rlines += [f"## {item['query_id']}", "", f"- user_query: {item['user_query']}", "",
               "| rank | score | doc_name | chunk_id | section_title | preview |",
               "|---:|---:|---|---|---|---|"]
    if item["results"]:
        for r in item["results"]:
            rlines.append(f"| {r['rank']} | {r['score']} | {esc(r['doc_name'])} | {esc(r['chunk_id'])} | "
                          f"{esc(r['section_title'])} | {esc(_preview(r['preview'], 80))} |")
    else:
        rlines.append("| - | - | (검색 결과 없음) | - | - | - |")
    rlines.append("")
save_markdown("outputs/day2/rag_test_results.md", "\n".join(rlines))
show_markdown("outputs/day2/rag_test_results.md")

## 6. State 구조 핵심 코드 실행

- **참조 원본 파일:** `src/day2/graph_state.py`
- **생성 산출물:** `outputs/day2/state_demo_result.md`

State는 **Node 사이에서 공유되는 업무 처리 문맥**입니다.
`retrieved_docs`는 **답변 생성과 grounding 검증의 핵심 입력**입니다.
이 구조는 3일차 **Multi-Agent Handoff State**로 확장될 수 있습니다.

아래 셀은 검색 전 State와 검색 후 State를 비교해 retrieved_docs가 어떻게 채워지는지 확인합니다.

In [ ]:
# State 구조 정의 및 변화 예시 (원본 graph_state.py 흐름 재구성)
def create_initial_state(user_query):
    """그래프 시작 시점의 State. user_query만 채우고 나머지는 빈 값으로 시작합니다."""
    return {"user_query": user_query, "rewritten_query": "", "equipment_id": "", "alarm_code": "",
            "retrieved_docs": [], "draft_answer": "", "final_answer": "", "grounding_status": "not_checked",
            "needs_rewrite": False, "retry_count": 0, "errors": [], "trace": []}


def add_trace(state, node_name, status, message, input_summary="", output_summary=""):
    """Node 실행 기록을 State.trace에 한 줄로 추가합니다(4일차 품질 평가의 기반)."""
    state["trace"].append({"node_name": node_name, "status": status, "message": message,
                           "input_summary": input_summary, "output_summary": output_summary})
    return state


def extract_alarm_code(text):
    m = re.search(r"ALM-[A-Z]+-[0-9]+", text or "")
    return m.group(0) if m else ""


def extract_equipment_id(text):
    m = re.search(r"EQP-[A-Z]+-[0-9]+", text or "")
    return m.group(0) if m else ""


def summarize_docs(docs):
    if not docs:
        return "검색된 문서 조각이 없습니다."
    return "\n".join(
        f"{d['rank']}. {d['doc_name']} / {d['section_title']} / score={d['score']} / {_preview(d['preview'], 80)}"
        for d in docs
    )


DEMO_QUERY = "EQP-EV-03에서 ALM-TEMP-402가 반복 발생했는데 원인 후보와 품질 영향 확인 항목을 알려줘"
state = create_initial_state(DEMO_QUERY)
print("검색 전 State:", {"equipment_id": state["equipment_id"], "alarm_code": state["alarm_code"],
                      "retrieved_docs": len(state["retrieved_docs"]), "grounding_status": state["grounding_status"]})

state["equipment_id"] = extract_equipment_id(DEMO_QUERY)
state["alarm_code"] = extract_alarm_code(DEMO_QUERY)
state["retrieved_docs"] = search_top_k_nb(DEMO_QUERY, 3)
state["grounding_status"] = "grounded" if state["retrieved_docs"] else "NEEDS_REWRITE"
print("검색 후 State:", {"equipment_id": state["equipment_id"], "alarm_code": state["alarm_code"],
                      "retrieved_docs": len(state["retrieved_docs"]), "grounding_status": state["grounding_status"]})

slines = ["# Day2 LangGraph State 구조 데모", "", "## 1. State 역할 요약", "",
          "- State는 Node 사이에서 공유되는 업무 처리 문맥입니다.",
          "- retrieved_docs는 답변 생성과 grounding 검증의 핵심 입력입니다.", "",
          "## 2. 현재 State 요약", "",
          f"- user_query: {state['user_query']}",
          f"- equipment_id: {state['equipment_id']}",
          f"- alarm_code: {state['alarm_code']}",
          f"- grounding_status: {state['grounding_status']}",
          f"- retrieved_docs 수: {len(state['retrieved_docs'])}", "",
          "## 3. retrieved_docs 요약", "", summarize_docs(state["retrieved_docs"]), ""]
save_markdown("outputs/day2/state_demo_result.md", "\n".join(slines))
show_markdown("outputs/day2/state_demo_result.md")

## 7. Node와 조건부 분기 핵심 코드 실행 (LangGraph 기반)

- **참조 원본 파일:** `src/day2/langgraph_rag_graph_runner.py`
- **생성 산출물:** `outputs/day2/langgraph_rag_trace.md`

이 섹션은 실제 **LangGraph의 StateGraph**를 사용해 RAG Agent v1의 Node 흐름과 조건부 분기를 구성합니다.

- Node는 **State를 입력받아 갱신하는 Agent 실행 단위**입니다.
- 조건부 분기는 **근거 없는 답변을 막기 위한 품질 관리 구조**입니다.
- Trace는 **Node 실행 순서와 State 변화를 확인하는 실행 검토 자료**이며, **4일차 실행 품질 평가의 기반**입니다.
- 실제 외부 LLM 호출은 사용하지 않고, **검색 근거 기반 교육용 답변 초안**을 생성합니다.

**그래프 흐름**

```
parse_query_node → retrieve_docs_node → generate_answer_node → verify_grounding_node → 조건부 분기
   - grounded: END
   - insufficient: query_rewrite_node → retrieve_docs_node (재검색)
```

> **안전 처리:** `langgraph` 패키지가 설치되어 있지 않거나 import/실행이 실패하면 `graph = None`으로 두고,
> **fallback final_state와 fallback trace**를 생성해 노트북 실행 흐름을 유지합니다(오류를 길게 디버깅하지 않습니다).

In [ ]:
# 실제 LangGraph StateGraph 기반 RAG Agent v1 Node/조건부 분기
# - 앞 섹션에서 정의한 create_initial_state / add_trace / extract_* / search_top_k_nb / summarize_docs / _preview / esc 를 재사용합니다.
from typing import TypedDict, List, Dict, Any


class ManufacturingRAGState(TypedDict):
    """모든 Node가 공유하는 업무 처리 문맥(State)의 스키마입니다.
    각 Node는 이 State를 읽어 필요한 값을 채운 뒤 다음 Node로 넘깁니다."""
    user_query: str
    rewritten_query: str
    equipment_id: str
    alarm_code: str
    retrieved_docs: List[Dict[str, Any]]
    draft_answer: str
    final_answer: str
    grounding_status: str
    needs_rewrite: bool
    retry_count: int
    errors: List[str]
    trace: List[Dict[str, str]]


MAX_RETRY_COUNT = 1  # 무한 재작성을 막기 위한 재검색 시도 상한입니다.


def parse_query_node(state):
    # 이 Node는 State의 질문에서 설비 ID/알람 코드를 추출해 다음 Node가 사용할 검색 조건을 채웁니다.
    query = state.get("rewritten_query") or state.get("user_query", "")
    state["equipment_id"] = extract_equipment_id(query)
    state["alarm_code"] = extract_alarm_code(query)
    add_trace(state, "parse_query_node", "success", "질문에서 설비 ID와 알람 코드를 추출했습니다.",
              query, f"equipment_id={state['equipment_id'] or '없음'}, alarm_code={state['alarm_code'] or '없음'}")
    return state


def retrieve_docs_node(state):
    # retrieved_docs는 답변 생성과 grounding 검증의 핵심 입력입니다. Top-3 근거 후보를 검색해 담습니다.
    query = state.get("rewritten_query") or state.get("user_query", "")
    docs = search_top_k_nb(query, 3)
    state["retrieved_docs"] = docs
    if docs:
        add_trace(state, "retrieve_docs_node", "success", "RAG 검색으로 Top-3 근거 후보를 확보했습니다.",
                  query, summarize_docs(docs))
    else:
        state["errors"].append("검색 결과가 없습니다.")
        add_trace(state, "retrieve_docs_node", "warning", "검색 결과가 없어 근거가 비어 있습니다.",
                  query, "retrieved_docs=0건")
    return state


def build_draft_answer(state):
    # 외부 LLM/API를 호출하지 않고 검색 근거만으로 만드는 교육용 답변 초안입니다.
    docs = state.get("retrieved_docs", [])
    if docs:
        evidence = "\n".join(f"- {d.get('doc_name', '')} / {d.get('section_title', '')} (score={d.get('score', '')})" for d in docs)
    else:
        evidence = "- 검색된 근거 문서가 없습니다."
    return ("## 근거 기반 답변 초안\n"
            f"- 설비 ID: {state.get('equipment_id', '') or '미확인'}\n"
            f"- 알람 코드: {state.get('alarm_code', '') or '미확인'}\n\n"
            "### 사용한 검색 근거\n" + evidence + "\n\n"
            "### 정리\n"
            "검색된 문서 근거를 기준으로 원인 후보와 품질 영향 확인 항목을 정리합니다. "
            "원인은 단정하지 않고 담당자 검토가 필요한 후보로만 제시합니다.")


def generate_answer_node(state):
    # 검색 근거 기반 교육용 답변 초안을 만들어 draft_answer/final_answer에 저장합니다(외부 LLM 미호출).
    answer = build_draft_answer(state)
    state["draft_answer"] = answer
    state["final_answer"] = answer
    status = "success" if state.get("retrieved_docs") else "warning"
    add_trace(state, "generate_answer_node", status, "검색 근거 기반 교육용 답변 초안을 생성했습니다(외부 LLM 미호출).",
              f"retrieved_docs={len(state.get('retrieved_docs', []))}건", "answer_source=notebook_draft")
    return state


def verify_grounding_node(state):
    # 근거 없는 답변을 그대로 내보내지 않게 막는 품질 관문입니다. 근거 유무로 grounding_status를 정합니다.
    docs = state.get("retrieved_docs", [])
    if docs:
        state["grounding_status"] = "grounded"
        state["needs_rewrite"] = False
        message = "검색 근거가 있어 grounding_status를 grounded로 설정했습니다."
    else:
        state["grounding_status"] = "insufficient"
        state["needs_rewrite"] = True
        message = "검색 근거가 부족하여 grounding_status를 insufficient로 설정하고 재작성이 필요합니다."
    add_trace(state, "verify_grounding_node", "success" if docs else "warning", message,
              f"retrieved_docs={len(docs)}건, retry_count={state.get('retry_count', 0)}",
              f"grounding_status={state['grounding_status']}, needs_rewrite={state['needs_rewrite']}")
    return state


def query_rewrite_node(state):
    # query_rewrite_node는 검색 근거가 부족할 때 재검색 가능한 질의로 변환하는 역할을 합니다.
    parts = [state.get("equipment_id", ""), state.get("alarm_code", ""), "온도 상승", "반복 알람", "품질 영향", "원인 후보"]
    rewritten = " ".join(p for p in parts if p).strip() or "온도 상승 반복 알람 품질 영향 원인 후보 확인 항목"
    state["rewritten_query"] = rewritten
    state["retry_count"] = state.get("retry_count", 0) + 1
    state["needs_rewrite"] = False
    add_trace(state, "query_rewrite_node", "success", "검색 근거 부족으로 핵심 키워드 중심으로 질의를 재작성했습니다.",
              state.get("user_query", ""), rewritten)
    return state


def route_after_grounding(state):
    # 조건부 분기: 근거가 충분하면 종료, 재시도 한도에 닿으면 종료, 그 외 근거 부족이면 재작성으로 보냅니다.
    if state.get("grounding_status") == "grounded":
        return "end"
    if state.get("retry_count", 0) >= MAX_RETRY_COUNT:
        return "end"
    if state.get("needs_rewrite") is True:
        return "rewrite"
    return "end"


# --- 실제 LangGraph StateGraph 구성 (설치되어 있으면 graph 생성, 아니면 graph=None) ---
graph = None
try:
    from langgraph.graph import StateGraph, END

    builder = StateGraph(ManufacturingRAGState)
    builder.add_node("parse_query", parse_query_node)
    builder.add_node("retrieve_docs", retrieve_docs_node)
    builder.add_node("generate_answer", generate_answer_node)
    builder.add_node("verify_grounding", verify_grounding_node)
    builder.add_node("query_rewrite", query_rewrite_node)
    builder.set_entry_point("parse_query")
    builder.add_edge("parse_query", "retrieve_docs")
    builder.add_edge("retrieve_docs", "generate_answer")
    builder.add_edge("generate_answer", "verify_grounding")
    # verify_grounding 이후 조건부 분기: grounded면 END, 부족하면 query_rewrite로 재검색 루프
    builder.add_conditional_edges("verify_grounding", route_after_grounding,
                                  {"rewrite": "query_rewrite", "end": END})
    builder.add_edge("query_rewrite", "retrieve_docs")
    graph = builder.compile()
    print("LangGraph StateGraph 컴파일 완료 (graph 생성됨)")
except Exception as exc:
    graph = None
    display(Markdown(
        f"> LangGraph를 사용할 수 없어 `graph=None`으로 둡니다(`{type(exc).__name__}`). "
        f"fallback final_state와 fallback trace로 노트북 실행 흐름을 유지합니다."
    ))


def build_fallback_state(user_query, note):
    # fallback final_state는 LangGraph 환경 문제로 invoke가 실패해도 이후 최종 결과 생성 흐름을 유지하기 위한 안전 장치입니다.
    state = create_initial_state(user_query)
    parse_query_node(state)
    retrieve_docs_node(state)
    generate_answer_node(state)
    verify_grounding_node(state)
    state["errors"].append(note)
    add_trace(state, "fallback", "warning", note, user_query,
              f"grounding_status={state.get('grounding_status', '')}")
    return state


# --- 그래프 실행: graph.invoke 실패해도 fallback으로 흐름을 잇습니다 ---
initial_state = create_initial_state(DEMO_QUERY)
if graph is not None:
    try:
        final_state = graph.invoke(initial_state)
    except Exception as exc:
        display(Markdown(f"> graph.invoke 실패(`{type(exc).__name__}`) → fallback final_state로 진행합니다."))
        final_state = build_fallback_state(DEMO_QUERY, f"graph.invoke 실패: {type(exc).__name__}")
else:
    final_state = build_fallback_state(DEMO_QUERY, "LangGraph 미설치/임포트 실패로 fallback trace를 생성했습니다.")

GRAPH_STATE = final_state  # 다음 섹션(8장 최종 결과)이 참조하는 변수 별칭을 유지합니다.

# --- Trace를 산출물로 저장하고 요약을 표시합니다 ---
tlines = ["# Day2 LangGraph RAG 실행 Trace", "",
          f"- LangGraph 사용: {graph is not None}", "",
          "## 1. 사용자 질문", "", f"- {final_state.get('user_query', '')}", "",
          "## 2. 최종 State 요약", "",
          f"- equipment_id: {final_state.get('equipment_id', '')}",
          f"- alarm_code: {final_state.get('alarm_code', '')}",
          f"- grounding_status: {final_state.get('grounding_status', '')}",
          f"- needs_rewrite: {final_state.get('needs_rewrite', False)}",
          f"- retry_count: {final_state.get('retry_count', 0)}",
          f"- retrieved_docs 수: {len(final_state.get('retrieved_docs', []))}", "",
          "## 3. Node 실행 Trace", "",
          "| 순서 | node_name | status | output_summary |", "|---:|---|---|---|"]
for i, tr in enumerate(final_state.get("trace", []), 1):
    tlines.append(f"| {i} | {tr.get('node_name', '')} | {tr.get('status', '')} | {esc(tr.get('output_summary', ''))} |")
tlines.append("")
save_markdown("outputs/day2/langgraph_rag_trace.md", "\n".join(tlines))

print("grounding_status:", final_state.get("grounding_status", ""))
print("Node 실행 순서:", [t.get("node_name", "") for t in final_state.get("trace", [])])
show_markdown("outputs/day2/langgraph_rag_trace.md")

## 7-1. LangGraph 구조 시각화

- 이 셀은 실제 LangGraph로 구성한 RAG Agent v1의 Node 흐름을 시각적으로 확인하는 **선택 시각화 셀**입니다.
- Node 실행 로직은 이전 셀(7장)에서 수행합니다.
- 이 셀은 `graph` 객체가 정상적으로 생성되었을 때 **Mermaid PNG**를 출력합니다.
- LangGraph 렌더링 환경 문제로 실패할 경우 **Markdown 기반 흐름도**로 대체합니다.

In [ ]:
# 7-1. LangGraph 구조 시각화 (선택 시각화 셀)
# - 이 셀은 RAG Agent v1의 Node 실행 흐름을 시각적으로 확인하기 위한 선택 시각화 셀입니다.
# - Node는 State를 입력받아 갱신하는 Agent 실행 단위입니다.
# - 조건부 분기는 근거 없는 답변을 막기 위한 품질 관리 구조입니다.
# - LangGraph 렌더링 환경이 없으면 Trace와 Markdown 흐름도로 대체합니다.
# - Mermaid PNG 생성 실패는 코드 오류라기보다 렌더링 환경 문제일 수 있습니다.
from IPython.display import Image, Markdown, display

_fallback_flow = """**LangGraph 구조 (Markdown 흐름도)**

```
parse_query_node
→ retrieve_docs_node
→ generate_answer_node
→ verify_grounding_node
→ 조건부 분기
   ├─ grounding 충분: END
   └─ grounding 부족: query_rewrite_node → retrieve_docs_node 재검색
```
"""

# graph 변수 존재 여부 + None 여부 + get_graph 메서드 여부를 모두 안전하게 확인합니다.
graph_obj = globals().get("graph", None)
try:
    if graph_obj is not None and hasattr(graph_obj, "get_graph"):
        display(Image(graph_obj.get_graph().draw_mermaid_png()))
    else:
        display(Markdown("graph 객체가 없어 Markdown 흐름도로 대체합니다."))
        display(Markdown(_fallback_flow))
except Exception as exc:
    # draw_mermaid_png 실패는 렌더링 환경 문제일 수 있으므로 Markdown 흐름도로 대체합니다.
    display(Markdown(f"LangGraph 그래프 이미지 출력 중 문제가 발생하여 Markdown 흐름도로 대체합니다. (`{type(exc).__name__}`)"))
    display(Markdown(_fallback_flow))

## 8. Day2 RAG Agent v1 최종 실행 구조

- **참조 원본 파일:** `src/day2/day2_rag_agent_v1.py`
- **생성 산출물:** `outputs/day2/day2_rag_agent_v1_result.md`

`day2_rag_agent_v1.py`는 모든 기능을 직접 구현하는 파일이 아닙니다.
**검색 모듈과 Graph 실행 모듈을 호출해 RAG Agent v1의 최종 State와 근거 기반 답변 결과를 정리하는 통합 실행 파일**입니다.

최종 결과는 문장이 자연스러운지보다,
**어떤 문서 근거를 사용했고 State와 Trace에 남았는지** 확인하는 것이 중요합니다.

In [ ]:
# 앞 단계에서 만든 최종 State를 사람이 읽을 리포트로 정리합니다.
FINAL_STATE = GRAPH_STATE
docs = FINAL_STATE.get("retrieved_docs", [])

flines = ["# Day2 RAG Agent v1 실행 결과 (노트북 통합 실행)", "",
          "## 1. 실행 개요", "",
          "- 검색 → State → Node/조건부 분기 → Trace → 최종 답변 흐름을 하나로 묶은 결과입니다.",
          f"- 검색 방식: {'Chroma Vector DB (의미 기반)' if CHROMA_READY else '키워드 기반 fallback'}", "",
          "## 2. 사용자 질문", "", f"- {FINAL_STATE['user_query']}", "",
          "## 3. 설비 ID / 알람 코드", "",
          f"- equipment_id: {FINAL_STATE['equipment_id']}",
          f"- alarm_code: {FINAL_STATE['alarm_code']}", "",
          "## 4. 검색 근거 (retrieved_docs)", "",
          "| rank | score | doc_name | chunk_id | section_title |", "|---:|---:|---|---|---|"]
if docs:
    for d in docs:
        flines.append(f"| {d['rank']} | {d['score']} | {esc(d['doc_name'])} | {esc(d['chunk_id'])} | {esc(d['section_title'])} |")
else:
    flines.append("| - | - | (근거 없음) | - | - |")
flines += ["", "## 5. grounding_status", "", f"- {FINAL_STATE['grounding_status']}", "",
           "## 6. 최종 답변 (초안)", "", FINAL_STATE.get("final_answer", ""), "",
           "## 7. Trace 요약", "", "| 순서 | node_name | status |", "|---:|---|---|"]
for i, tr in enumerate(FINAL_STATE["trace"], 1):
    flines.append(f"| {i} | {tr['node_name']} | {tr['status']} |")
flines += ["", "> 핵심은 답변 문장이 매끄러운지가 아니라, 어떤 문서 근거를 썼고 State·Trace에 남았는지입니다.", ""]

save_markdown("outputs/day2/day2_rag_agent_v1_result.md", "\n".join(flines))
show_markdown("outputs/day2/day2_rag_agent_v1_result.md")

## 설계 질문

1. chunk 크기가 너무 크면 어떤 문제가 생길까요?
2. metadata가 잘못 추출되면 어떤 검색 품질 문제가 생길까요?
3. Top-3 검색 결과 중 score가 높지만 본문 근거가 약하면 Agent는 답변해야 할까요?
4. 검색 결과가 부족할 때 `query_rewrite_node`는 어떤 기준으로 동작해야 할까요?
5. 오늘 만든 RAG 검색 기능을 `search_manual` MCP Tool로 분리한다면 input/output schema는 어떻게 잡아야 할까요?

## 3일차 MCP Tool 연결

오늘 만든 RAG 검색 기능은 3일차에 **`search_manual` MCP Tool**로 분리됩니다.

즉, Agent 내부 코드에서 직접 검색하던 기능을
**Tool Contract 기반으로 호출 가능한 기능**으로 확장합니다.
이후 제조 DB/Log Tool과 함께 **MCP 방식으로 호출**됩니다.

**연결 관계**

| 2일차 구조 | 3일차 이후 확장 |
|---|---|
| `rag_search.py` | → `search_manual` MCP Tool |
| metadata | → Tool 입력값 및 검색 필터 |
| State | → Multi-Agent Handoff State |
| Trace | → MCP Call Trace 및 4일차 실행 품질 평가 |